# SkyGuard Weather Anomaly Detection, Multi-Source Evidence Fusion & Root Cause Analysis Pipeline

This notebook implements the complete, production-grade **SkyGuard Target Architecture** for multi-station weather sensor telemetry:

```
                         ┌──────────────────────┐
                         │     AWS TELEMETRY    │
                         │ Temperature / RH / P │
                         │ Station / Timestamp  │
                         └───────────┬──────────┘
                                     │
                                     ▼
                         ┌──────────────────────┐
                         │   DATA QUALITY LAYER │
                         │                      │
                         │ • Missing values     │
                         │ • Invalid values     │
                         │ • Timestamp checks   │
                         │ • Station validation │
                         └───────────┬──────────┘
                                     │
                                     ▼
                         ┌──────────────────────┐
                         │ FEATURE ENGINEERING  │
                         │                      │
                         │ • Lag features       │
                         │ • Rolling statistics │
                         │ • Rate of change     │
                         │ • Z-scores           │
                         │ • Spatial features   │
                         │ • Cross-sensor       │
                         │ • Dew-point features │
                         └───────────┬──────────┘
                                     │
                ┌────────────────────┼────────────────────┐
                │                    │                    │
                ▼                    ▼                    ▼
       ┌────────────────┐   ┌────────────────┐   ┌────────────────┐
       │ ISOLATION      │   │    XGBOOST     │   │    PHYSICS     │
       │ FOREST         │   │                │   │  CONSISTENCY   │
       │                │   │ Root Cause     │   │                │
       │ Novelty Score  │   │ Classification │   │ • Range        │
       │                │   │                │   │ • Rate         │
       └───────┬────────┘   └───────┬────────┘   │ • T/RH         │
               │                    │            │ • Pressure     │
               │                    │            │ • Cross-sensor │
               │                    │            └───────┬────────┘
               │                    │                    │
               │            ┌───────┴──────────┐         │
               │            │                  │         │
               │            ▼                  ▼         │
               │      XGBoost Class       Confidence     │
               │                                         │
               └────────────────┬────────────────────────┘
                                │
                                ▼
                    ┌─────────────────────────┐
                    │     EVIDENCE FUSION     │
                    │                         │
                    │ IF novelty              │
                    │ + temporal evidence     │
                    │ + spatial evidence      │
                    │ + physics evidence      │
                    │ + classifier evidence   │
                    └────────────┬────────────┘
                                 │
                                 ▼
                    ┌─────────────────────────┐
                    │     DECISION ENGINE     │
                    └────────────┬────────────┘
                                 │
               ┌─────────────────┼──────────────────┐
               │                 │                  │
               ▼                 ▼                  ▼
          ┌─────────┐      ┌──────────────┐   ┌──────────────┐
          │ NORMAL  │      │KNOWN ANOMALY │   │NOVEL ANOMALY │
          └─────────┘      └───────┬──────┘   └───────┬──────┘
                                   │                   │
                                   ▼                   │
                          ┌─────────────────┐          │
                          │   ROOT CAUSE    │          │
                          │                 │          │
                          │ • Temp spike    │          │
                          │ • Humidity      │          │
                          │ • Pressure      │          │
                          │ • Freeze        │          │
                          │ • Drift         │          │
                          │ • Offset        │          │
                          │ • Missing       │          │
                          │ • Multivariate  │          │
                          │ • Spatial       │          │
                          └────────┬────────┘          │
                                   │                   │
                                   └─────────┬─────────┘
                                             ▼
                                  ┌────────────────────┐
                                  │   SEVERITY ENGINE  │
                                  │                    │
                                  │ LOW                │
                                  │ MEDIUM             │
                                  │ HIGH               │
                                  │ CRITICAL           │
                                  └──────────┬─────────┘
                                             │
                                             ▼
                                  ┌────────────────────┐
                                  │  SENSOR HEALTH     │
                                  │      SCORE         │
                                  │                    │
                                  │ 0 ─────────── 100  │
                                  └──────────┬─────────┘
                                             │
                                             ▼
                                  ┌────────────────────┐
                                  │ SHAP EXPLANATION   │
                                  │                    │
                                  │ Why was it flagged?│
                                  │ What features      │
                                  │ contributed?       │
                                  └──────────┬─────────┘
                                             │
                                             ▼
                                  ┌────────────────────┐
                                  │ STRUCTURED REPORT  │
                                  │                    │
                                  │ Station            │
                                  │ Time               │
                                  │ Evidence           │
                                  │ Root cause         │
                                  │ Confidence         │
                                  │ Severity           │
                                  │ Health             │
                                  │ SHAP factors       │
                                  └──────────┬─────────┘
                                             │
                                             ▼
                                  ┌────────────────────┐
                                  │   LLM REPORTING    │
                                  │      LAYER         │
                                  │                    │
                                  │ Explain + Summarize│
                                  │ + Recommend        │
                                  └──────────┬─────────┘
                                             │
                                             ▼
                                  ┌────────────────────┐
                                  │ MAINTENANCE ACTION │
                                  │                    │
                                  │ Inspect            │
                                  │ Calibrate          │
                                  │ Compare neighbors  │
                                  │ Replace sensor     │
                                  └──────────┬─────────┘
                                             │
                                             ▼
                                  ┌────────────────────┐
                                  │ OPERATOR FEEDBACK  │
                                  │                    │
                                  │ Confirmed          │
                                  │ False alarm        │
                                  │ Corrected cause    │
                                  │ Comments           │
                                  └──────────┬─────────┘
                                             │
                                             ▼
                                  ┌────────────────────┐
                                  │ MODEL IMPROVEMENT  │
                                  │                    │
                                  │ Validated labels   │
                                  │ Evaluation         │
                                  │ Retraining         │
                                  └────────────────────┘
```

### Complete Pipeline Architecture & Key Principles:
1. **Data Quality Layer**: Ingestion, validation, missing value imputation, and chronological alignment per station.
2. **Feature Engineering**: Lags, rolling statistics, rate-of-change deltas, psychrometric dew point depression, and regional cluster consensus.
3. **Physics Consistency Layer**: Meteorological ranges, rate-of-change limits, psychrometric/dew-point depression consistency, and cross-sensor physical coupling.
4. **Stage 1 Isolation Forest Novelty**: Continuous unsupervised novelty scoring on clean telemetry without hard gating.
5. **10-Class XGBoost Classifier**: Balanced multi-class root cause classification across normal and 9 physical failure modes.
6. **Evidence Fusion Layer**: Multi-source weighted fusion combining IF novelty, temporal/statistical evidence, spatial consensus, physics consistency, and XGBoost confidence.
7. **Decision Engine**: 3-way triage (`normal`, `known_anomaly`, `novel_anomaly`) preventing forced misclassification of unmodeled failure modes.
8. **Configurable Severity Engine**: Multi-factor severity scoring mapped to operational tiers (`LOW`, `MEDIUM`, `HIGH`, `CRITICAL`).
9. **Transparent 0–100 Sensor Health Score**: Itemized penalty deductions reflecting anomaly recurrence, drift, dropouts, and physics violations.
10. **Human-Readable SHAP Attributions**: Natural language diagnostic explanation of primary sensor drivers.
11. **Structured LLM Diagnostic Object & LLM Reporting**: Machine-readable JSON summary and automated prompt briefing.
12. **Maintenance Recommendation Engine**: Domain-grounded physical maintenance and calibration action items.
13. **Operator Feedback & Safe Retraining**: Feedback schema with human validation quarantine to safely improve models without label poisoning.

In [ ]:
# MODIFIED: Imports, Visual Configurations, Class Taxonomy, and Architecture Configs
import os
import json
import warnings
import joblib
from datetime import datetime
from typing import Dict, Any, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve
)

# Advanced ML & Explainability
import xgboost as xgb
import shap

# Plotting configurations
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
warnings.filterwarnings('ignore')

# ------------------------------------------------------------
# 10-Class Label Definitions (0: Normal, 1..9: Anomaly Modes)
# ------------------------------------------------------------
ALL_CLASSES = [
    "normal",
    "temperature_spike",
    "humidity_spike",
    "pressure_jump",
    "freeze",
    "drift",
    "offset",
    "missing_data",
    "multivariate_inconsistency",
    "spatial_inconsistency"
]

ANOMALY_CLASSES = ALL_CLASSES[1:]
NUM_CLASSES = len(ALL_CLASSES)

# Mappings for 10-Class Classifier (0-indexed contiguous)
CLASS_TO_IDX = {name: idx for idx, name in enumerate(ALL_CLASSES)}
IDX_TO_CLASS = {idx: name for idx, name in enumerate(ALL_CLASSES)}

# ------------------------------------------------------------
# NEW: Architecture Configuration Dictionaries
# ------------------------------------------------------------

# 1. Physics Consistency Rules & Meteorological Limits
PHYSICS_CONFIG = {
    "temp_min_c": -40.0,
    "temp_max_c": 60.0,
    "humidity_min_pct": 0.0,
    "humidity_max_pct": 100.0,
    "pressure_min_hpa": 870.0,
    "pressure_max_hpa": 1085.0,
    "temp_max_rate_c_per_hr": 12.0,
    "humidity_max_rate_pct_per_hr": 35.0,
    "pressure_max_rate_hpa_per_hr": 6.0,
    "max_dewpoint_depression_c": 45.0,
    "min_dewpoint_depression_c": -0.5,
    "weight_range": 0.30,
    "weight_rate": 0.25,
    "weight_dewpoint": 0.25,
    "weight_cross_sensor": 0.20
}

# 2. Evidence Fusion Weights (Configurable across 5 Evidence Sources)
FUSION_CONFIG = {
    "w_iforest": 0.20,       # Isolation Forest Novelty
    "w_temporal": 0.20,      # Rolling Statistical & Persistence Evidence
    "w_spatial": 0.20,       # Regional Cluster Consensus Evidence
    "w_physics": 0.20,       # Physical & Meteorological Consistency
    "w_xgboost": 0.20        # Supervised Classifier Confidence Evidence
}

# 3. Decision Engine Calibration Thresholds
DECISION_CONFIG = {
    "anomaly_threshold": 0.45,       # Fused score >= 0.45 flags anomaly
    "known_class_threshold": 0.40,   # Minimum top-class probability for known anomaly
    "novelty_threshold": 0.65        # High IF novelty with low class confidence triggers novel_anomaly
}

# 4. Severity Scoring Weights & Tiers
SEVERITY_CONFIG = {
    "weight_anomaly_strength": 0.35,
    "weight_persistence": 0.25,
    "weight_confidence": 0.20,
    "weight_multi_sensor": 0.20,
    "thresholds": {
        "LOW": 0.00,
        "MEDIUM": 0.40,
        "HIGH": 0.65,
        "CRITICAL": 0.85
    }
}
# 5. Sensor Health Scoring Configuration (Base = 100)
HEALTH_CONFIG = {
    "base_score": 100.0,
    "max_anomaly_deduction": 35.0,
    "max_drift_deduction": 25.0,
    "max_missing_deduction": 20.0,
    "max_physics_deduction": 20.0
}

print(f"Configured complete 10-Class Target Space for XGBoost (num_classes={NUM_CLASSES}):")
for idx, name in enumerate(ALL_CLASSES):
    print(f"  Class {idx:2d}: {name}")
print("\nConfigured SkyGuard Target Architecture Parameters:")
print(f"  Evidence Fusion Weights: {FUSION_CONFIG}")
print(f"  Decision Engine Thresholds: {DECISION_CONFIG}")
print(f"  Severity Thresholds: {SEVERITY_CONFIG['thresholds']}")

## Step 1: Data Preprocessing & Domain Anomaly Injection Functions

In [ ]:
# PRESERVED & MODIFIED: Step 1 Preprocessing & Anomaly Injection Functions

def load_and_preprocess_raw(filepath: str) -> pd.DataFrame:
    """
    Loads raw weather data and performs standard cleaning:
    - Strips whitespace from column names
    - Coerces numeric sensor readings
    - Sorts chronologically per weather station
    - Handles missing sensor values via station-wise linear interpolation + ffill/bfill
    """
    df = pd.read_csv(filepath, low_memory=False)
    df.columns = df.columns.str.strip()

    numeric_cols = ['temperature_c', 'humidity_pct', 'pressure_hpa', 'latitude', 'longitude']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Chronological sort per station
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values(by=['station_id', 'timestamp']).reset_index(drop=True)

    # Station-wise interpolation for raw missing values
    sensor_cols = ['temperature_c', 'humidity_pct', 'pressure_hpa']
    df[sensor_cols] = df.groupby('station_id')[sensor_cols].transform(
        lambda group: group.interpolate(method='linear').ffill().bfill()
    )

    return df


def inject_anomalies(df: pd.DataFrame, random_state: int = 42) -> pd.DataFrame:
    """
    Injects 9 distinct, domain-realistic anomaly types onto raw physical weather readings.
    Sets anomaly_label (0 = normal, 1 = anomaly) and root_cause class (10 classes total).
    Uses station-aware windows and index sampling to preserve temporal integrity.
    """
    np.random.seed(random_state)
    df_injected = df.copy()
    df_injected['anomaly_label'] = 0
    df_injected['root_cause'] = 'normal'

    n = len(df_injected)

    def get_random_indices(pct=0.005):
        return np.random.choice(df_injected.index, size=int(n * pct), replace=False)

    def get_station_windows(window_size=11, num_windows=50):
        windows = []
        for station_id, station_df in df_injected.groupby('station_id', sort=False):
            station_indices = station_df.index.to_numpy()
            if len(station_indices) < window_size:
                continue
            valid_starts = np.arange(len(station_indices) - window_size + 1)
            for _ in range(num_windows):
                start_pos = np.random.choice(valid_starts)
                window_indices = station_indices[start_pos:start_pos + window_size]
                windows.append(window_indices)

        if len(windows) <= num_windows:
            return windows
        selected_indices = np.random.choice(len(windows), size=num_windows, replace=False)
        return [windows[i] for i in selected_indices]

    # 1. Temperature Spike: sudden unphysical heat jump (+15°C to +30°C)
    idx = get_random_indices(0.005)
    df_injected.loc[idx, 'temperature_c'] += np.random.uniform(15, 30, size=len(idx))
    df_injected.loc[idx, ['anomaly_label', 'root_cause']] = [1, 'temperature_spike']

    # 2. Humidity Spike: sudden surge (+40% capped at 100%)
    idx = get_random_indices(0.005)
    df_injected.loc[idx, 'humidity_pct'] = (df_injected.loc[idx, 'humidity_pct'] + 40).clip(0, 100)
    df_injected.loc[idx, ['anomaly_label', 'root_cause']] = [1, 'humidity_spike']

    # 3. Pressure Jump: sudden barometric jump (+10 to +20 hPa)
    idx = get_random_indices(0.005)
    df_injected.loc[idx, 'pressure_hpa'] += np.random.uniform(10, 20, size=len(idx))
    df_injected.loc[idx, ['anomaly_label', 'root_cause']] = [1, 'pressure_jump']

    # 4. Freeze: Stuck temperature sensor (constant reading for 11 consecutive hours)
    freeze_windows = get_station_windows(window_size=11, num_windows=50)
    for window_indices in freeze_windows:
        freeze_val = df_injected.loc[window_indices[0], 'temperature_c']
        df_injected.loc[window_indices, 'temperature_c'] = freeze_val
        df_injected.loc[window_indices, ['anomaly_label', 'root_cause']] = [1, 'freeze']

    # 5. Drift: Gradual calibration degradation (+0°C to +10°C over 21 consecutive hours)
    drift_windows = get_station_windows(window_size=21, num_windows=50)
    for window_indices in drift_windows:
        drift_val = np.linspace(0, 10, 21)
        df_injected.loc[window_indices, 'temperature_c'] += drift_val
        df_injected.loc[window_indices, ['anomaly_label', 'root_cause']] = [1, 'drift']

    # 6. Offset: Constant sensor bias (+5.0°C)
    idx = get_random_indices(0.005)
    df_injected.loc[idx, 'temperature_c'] += 5.0
    df_injected.loc[idx, ['anomaly_label', 'root_cause']] = [1, 'offset']

    # 7. Missing Data / Outage: NaN sensor dropout per station
    idx = get_random_indices(0.005)
    df_injected.loc[idx, ['temperature_c', 'humidity_pct']] = np.nan
    df_injected.loc[idx, ['anomaly_label', 'root_cause']] = [1, 'missing_data']

    # 8. Multivariate Inconsistency: High temp (45°C) + Extreme Humidity (95%)
    idx = get_random_indices(0.005)
    df_injected.loc[idx, 'temperature_c'] = 45.0
    df_injected.loc[idx, 'humidity_pct'] = 95.0
    df_injected.loc[idx, ['anomaly_label', 'root_cause']] = [1, 'multivariate_inconsistency']

    # 9. Spatial Inconsistency: Station deviates +15°C from regional cluster mean
    raw_cluster_mean = df_injected.groupby(['cluster', 'timestamp'])['temperature_c'].transform('mean')
    idx = get_random_indices(0.005)
    df_injected.loc[idx, 'temperature_c'] = raw_cluster_mean.loc[idx] + 15.0
    df_injected.loc[idx, ['anomaly_label', 'root_cause']] = [1, 'spatial_inconsistency']

    return df_injected


# NEW: Helper function to inject completely novel, unmodeled anomalies (e.g., high-frequency sinusoidal oscillation)
def inject_novel_anomalies(df: pd.DataFrame, random_state: int = 99, num_instances: int = 40) -> pd.DataFrame:
    """
    Injects an unseen, novel anomaly pattern (unmodeled high-frequency oscillation / erratic cross-sensor inversion)
    for validating the Novel Anomaly Detection component of the Decision Engine.
    """
    np.random.seed(random_state)
    df_novel = df.copy()
    sample_idx = np.random.choice(df_novel.index, size=num_instances, replace=False)
    # Unmodeled pattern: high-frequency alternating sine oscillation with pressure anti-coupling
    t_step = np.arange(len(sample_idx))
    df_novel.loc[sample_idx, 'temperature_c'] += 18.0 * np.sin(2 * np.pi * t_step / 3.0)
    df_novel.loc[sample_idx, 'pressure_hpa'] -= 15.0 * np.cos(2 * np.pi * t_step / 3.0)
    df_novel.loc[sample_idx, 'anomaly_label'] = 1
    df_novel.loc[sample_idx, 'root_cause'] = 'novel_anomaly'
    return df_novel

## Step 2: Unified Feature Engineering Function (`engineer_features`)

Calculates spatial cluster consensus, cyclical time, multivariate interactions, psychrometric dew point calculations, and station temporal dynamics.

In [ ]:
# MODIFIED: Unified Feature Engineering Function with Psychrometric & Temporal Dynamics

def calculate_dewpoint_magnus(temp_c: pd.Series, humidity_pct: pd.Series) -> pd.Series:
    """
    Calculates dew point temperature using the standard Magnus-Tetens approximation formula:
    gamma(T, RH) = (a * T / (b + T)) + ln(RH / 100)
    T_dew = (b * gamma) / (a - gamma)
    where a = 17.27, b = 237.7 deg C.
    """
    a = 17.27
    b = 237.7
    rh_safe = humidity_pct.clip(1.0, 100.0) / 100.0
    t_safe = temp_c.clip(-50.0, 60.0)
    gamma = (a * t_safe) / (b + t_safe) + np.log(rh_safe)
    dewpoint = (b * gamma) / (a - gamma)
    return dewpoint


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Unified feature engineering function applied to both clean and injected datasets.
    Calculates all spatial, temporal, rolling, lag, cross-sensor, and psychrometric features.
    """
    df = df.copy()

    # Ensure timestamp is datetime and sorted chronologically per station
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values(by=['station_id', 'timestamp']).reset_index(drop=True)

    # 1. Temporal Cyclical Features
    df['hour'] = df['timestamp'].dt.hour
    df['month'] = df['timestamp'].dt.month
    df['day'] = df['timestamp'].dt.day
    df['dayofweek'] = df['timestamp'].dt.dayofweek
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24.0)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24.0)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12.0)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12.0)

    # 2. Spatial & Cluster Consensus Features
    df['cluster_temp_mean'] = df.groupby(['cluster', 'timestamp'])['temperature_c'].transform('mean')
    df['cluster_temp_std'] = df.groupby(['cluster', 'timestamp'])['temperature_c'].transform('std').fillna(1e-5).replace(0.0, 1e-5)

    df['cluster_press_mean'] = df.groupby(['cluster', 'timestamp'])['pressure_hpa'].transform('mean')
    df['cluster_press_std'] = df.groupby(['cluster', 'timestamp'])['pressure_hpa'].transform('std').fillna(1e-5).replace(0.0, 1e-5)

    df['cluster_hum_mean'] = df.groupby(['cluster', 'timestamp'])['humidity_pct'].transform('mean')
    df['cluster_hum_std'] = df.groupby(['cluster', 'timestamp'])['humidity_pct'].transform('std').fillna(1e-5).replace(0.0, 1e-5)

    # Spatial differences and Z-scores
    df['spatial_temp_diff'] = df['temperature_c'] - df['cluster_temp_mean']
    df['spatial_temp_zscore'] = df['spatial_temp_diff'] / df['cluster_temp_std']

    df['spatial_press_diff'] = df['pressure_hpa'] - df['cluster_press_mean']
    df['spatial_press_zscore'] = df['spatial_press_diff'] / df['cluster_press_std']

    df['spatial_hum_diff'] = df['humidity_pct'] - df['cluster_hum_mean']
    df['spatial_hum_zscore'] = df['spatial_hum_diff'] / df['cluster_hum_std']

    # Neighbor deviation & consistency scores
    df['neighbor_dev_score'] = df['spatial_temp_diff'].abs() / (df['cluster_temp_mean'].abs() + 1e-5)
    df['station_neighbor_consistency'] = 1.0 / (1.0 + df['neighbor_dev_score'])
    df['cluster_anomaly_pct'] = df.groupby(['cluster', 'timestamp'])['spatial_temp_zscore'].transform(lambda x: (x.abs() > 2.0).mean())

    # 3. Multivariate Ratios & Interactions
    df['temp_press_ratio'] = df['temperature_c'] / (df['pressure_hpa'] + 1e-5)
    df['temp_hum_ratio'] = df['temperature_c'] / (df['humidity_pct'] + 1e-5)
    df['hum_press_ratio'] = df['humidity_pct'] / (df['pressure_hpa'] + 1e-5)

    # NEW: Psychrometric Dew Point & Dew Point Depression Features
    df['dewpoint_c'] = calculate_dewpoint_magnus(df['temperature_c'], df['humidity_pct'])
    df['dewpoint_depression_c'] = df['temperature_c'] - df['dewpoint_c']
    df['vapor_pressure_ratio'] = df['humidity_pct'] * np.exp(17.27 * df['temperature_c'] / (237.7 + df['temperature_c'])) / 100.0

    # 4. Temporal Station-Level Lag, Delta, and Rolling Features
    target_sensors = ['temperature_c', 'humidity_pct', 'pressure_hpa']
    new_features = []

    for col in target_sensors:
        grouped = df.groupby('station_id')[col]

        # Lags
        lag1 = grouped.shift(1).rename(f'{col}_lag1')
        lag24 = grouped.shift(24).rename(f'{col}_lag24')

        # First derivatives / Rate of change per hour
        diff_lag1 = (df[col] - lag1).rename(f'{col}_diff_lag1')
        diff_lag24 = (df[col] - lag24).rename(f'{col}_diff_lag24')
        rate_of_change_1h = diff_lag1.abs().rename(f'{col}_rate_1h')

        # 6-Hour rolling statistics
        roll6_med = grouped.transform(lambda x: x.rolling(6, min_periods=1).median()).rename(f'{col}_roll6_med')
        roll6_mean = grouped.transform(lambda x: x.rolling(6, min_periods=1).mean()).rename(f'{col}_roll6_mean')
        roll6_var = grouped.transform(lambda x: x.rolling(6, min_periods=1).var()).fillna(0.0).rename(f'{col}_roll6_var')

        def consecutive_identical_count(series):
            values = series.to_numpy()
            counts = np.ones(len(values), dtype=int)
            for i in range(1, len(values)):
                if pd.notna(values[i]) and pd.notna(values[i - 1]) and values[i] == values[i - 1]:
                    counts[i] = counts[i - 1] + 1
                else:
                    counts[i] = 1
            return pd.Series(counts, index=series.index)

        frozen_count = grouped.transform(consecutive_identical_count).rename(f'{col}_frozen_count')

        # 24-Hour rolling baseline & local statistical z-scores
        roll24_mean = grouped.transform(lambda x: x.rolling(24, min_periods=1).mean()).rename(f'{col}_roll24_mean')
        roll24_std = grouped.transform(lambda x: x.rolling(24, min_periods=1).std()).fillna(1e-5).replace(0.0, 1e-5).rename(f'{col}_roll24_std')
        roll24_diff = (df[col] - roll24_mean).rename(f'{col}_roll24_diff')
        roll24_zscore = (roll24_diff.abs() / roll24_std).rename(f'{col}_roll24_zscore')

        new_features.extend([
            lag1, lag24, diff_lag1, diff_lag24, rate_of_change_1h,
            roll6_med, roll6_mean, roll6_var,
            frozen_count, roll24_mean,
            roll24_std, roll24_diff, roll24_zscore
        ])

    df_engineered = pd.concat([df] + new_features, axis=1)

    # Remove initial warm-up rows (first 24 hours per station) where 24h lag features are undefined
    station_cumcount = df_engineered.groupby('station_id').cumcount()
    df_engineered = df_engineered[station_cumcount >= 24].reset_index(drop=True)
    df_engineered = df_engineered.fillna(0.0)

    return df_engineered

## Step 3: Pipeline Execution - Feature Extraction on `df_clean` and `df_injected`

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

In [ ]:
# PRESERVED: Step 3 Data Loading & Feature Extraction Execution

# Determine dataset path (local or Google Colab)
if os.path.exists('skyguard_weather.csv'):
    data_path = 'skyguard_weather.csv'
elif os.path.exists('/content/drive/MyDrive/Datasets/skyguard_weather.csv'):
    data_path = '/content/drive/MyDrive/Datasets/skyguard_weather.csv'
elif os.path.exists('../skyguard_weather.csv'):
    data_path = '../skyguard_weather.csv'
else:
    data_path = 'skyguard_weather.csv'

print(f"Loading raw weather data from: {data_path}")
df_raw = load_and_preprocess_raw(data_path)
print(f"Raw Clean Data Shape: {df_raw.shape}")

# 1. Create clean baseline and injected test copies
df_clean_raw = df_raw.copy()
df_injected_raw = inject_anomalies(df_raw.copy(), random_state=42)

# 2. Apply feature engineering to BOTH datasets
print("Applying feature engineering to df_clean...")
df_clean = engineer_features(df_clean_raw)
df_clean['anomaly_label'] = 0
df_clean['root_cause'] = 'normal'

print("Applying feature engineering to df_injected...")
df_injected = engineer_features(df_injected_raw)

print("\n--- Dataset Summary ---")
print(f"df_clean shape: {df_clean.shape}")
print(f"df_injected shape: {df_injected.shape}")
print("\nGround Truth Distribution across All 10 Classes in df_injected:")
print(df_injected['root_cause'].value_counts())

## Step 4: Model Feature Matrix Preparation & Encoding

In [ ]:
# PRESERVED: Step 4 Model Feature Matrix Preparation & One-Hot Encoding

metadata_cols = [
    'station_id', 'station_name', 'city', 'timestamp',
    'root_cause', 'anomaly_label', 'hour', 'month',
    'iforest_pred', 'anomaly_score'
]

# One-Hot encode cluster column consistently
df_clean_encoded = pd.get_dummies(df_clean.drop(columns=metadata_cols, errors='ignore'), columns=['cluster'], drop_first=True)
bool_cols = df_clean_encoded.select_dtypes(include='bool').columns
df_clean_encoded[bool_cols] = df_clean_encoded[bool_cols].astype(int)

df_injected_encoded = pd.get_dummies(df_injected.drop(columns=metadata_cols, errors='ignore'), columns=['cluster'], drop_first=True)
bool_cols_inj = df_injected_encoded.select_dtypes(include='bool').columns
df_injected_encoded[bool_cols_inj] = df_injected_encoded[bool_cols_inj].astype(int)

# Align columns
df_injected_encoded = df_injected_encoded.reindex(columns=df_clean_encoded.columns, fill_value=0)
feature_cols = [col for col in df_clean_encoded.columns if col not in metadata_cols]

print(f"Total model feature count: {len(feature_cols)}")
print("Sample features:", feature_cols[:10])

## Step 5: Train Stage 1 Isolation Forest Anomaly Detector (Continuous Novelty Evidence, No Hard Gate)

- Trained **strictly on clean normal telemetry** (`df_clean`).
- Generates a **continuous unsupervised novelty score** ($E_{\text{novelty}} \in [0, 1]$).
- **No hard gate**: Records are never discarded or forced normal solely because Isolation Forest misses them; novelty contributes as one weighted channel in Evidence Fusion.

In [ ]:
# MODIFIED: Step 5 Isolation Forest Model Training & Continuous Novelty Scoring (No Hard Gate)

# 1. Initialize and Fit Isolation Forest on Normal Data
iso_model = IsolationForest(
    n_estimators=250,
    max_samples=512,
    contamination=0.03,
    random_state=42,
    n_jobs=-1
)

print("Training Isolation Forest on clean baseline telemetry...")
iso_model.fit(df_clean_encoded[feature_cols])

# 2. Predict Continuous Novelty Scores on df_injected
raw_scores = iso_model.decision_function(df_injected_encoded[feature_cols])
# Normalized ML anomaly score: 0 = completely normal, 1 = maximum anomalous novelty
ml_novelty_scores = 1.0 - ((raw_scores - raw_scores.min()) / (raw_scores.max() - raw_scores.min() + 1e-9))

# 3. Statistical Sensor Consensus Score (for standalone benchmark comparison)
df_injected = df_injected.reset_index(drop=True)
spatial_temp_zscore = df_injected['spatial_temp_zscore'].to_numpy()
temperature_roll24_zscore = df_injected['temperature_c_roll24_zscore'].to_numpy()
humidity_roll24_zscore = df_injected['humidity_pct_roll24_zscore'].to_numpy()
stat_scores = np.maximum.reduce([
    np.clip(np.abs(spatial_temp_zscore) / 3.0, 0, 1),
    np.clip(temperature_roll24_zscore / 3.0, 0, 1),
    np.clip(humidity_roll24_zscore / 3.0, 0, 1)
])

# Standalone IF ensemble score for unsupervised baseline evaluation
if_standalone_score = (0.65 * ml_novelty_scores + 0.35 * stat_scores)

# 4. Validation Threshold Tuning for Standalone Benchmark
threshold_indices = np.arange(len(df_injected))
validation_idx, test_idx = train_test_split(
    threshold_indices,
    test_size=0.25,
    random_state=42,
    stratify=df_injected['anomaly_label']
)

validation_scores = if_standalone_score[validation_idx]
validation_labels = df_injected['anomaly_label'].iloc[validation_idx].to_numpy()

best_thresh = 0.50
best_f1 = 0.0
for thresh in np.arange(0.20, 0.85, 0.02):
    validation_preds = (validation_scores >= thresh).astype(int)
    f1 = f1_score(validation_labels, validation_preds)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = thresh

# Record scores on df_injected for downstream layers
df_injected['iforest_ml_score'] = ml_novelty_scores
df_injected['iforest_pred'] = (if_standalone_score >= best_thresh).astype(int)
df_injected['anomaly_score'] = if_standalone_score

# 5. Standalone Binary Evaluation Metrics (Unsupervised Benchmark)
y_true = df_injected['anomaly_label']
y_pred = df_injected['iforest_pred']
roc_auc = roc_auc_score(y_true, if_standalone_score)
pr_auc = average_precision_score(y_true, if_standalone_score)

print(f"\n=== Isolation Forest Unsupervised Baseline Performance ===")
print(f"Optimal Standalone Threshold: {best_thresh:.2f}")
print(f"ROC-AUC Score: {roc_auc:.4f}")
print(f"PR-AUC Score:  {pr_auc:.4f}")
print(f"F1 Score:      {best_f1:.4f}")
print(f"Precision:     {precision_score(y_true, y_pred):.4f}")
print(f"Recall:        {recall_score(y_true, y_pred):.4f}")

## Step 6: Train XGBoost Multi-Class Classifier on All 10 Classes (Including Normal)

- Trained on **all 10 classes** (`normal` + 9 physical failure modes) with balanced class sample weights.
- Provides class probabilities and multi-class confidence vectors.

In [ ]:
# PRESERVED & MODIFIED: Step 6 XGBoost 10-Class Classifier Training & Evaluation

# Features and Target for ALL 10 Classes
X = df_injected_encoded[feature_cols]
y = df_injected['root_cause'].map(CLASS_TO_IDX)

# Ensure clean mapping with no NaNs
valid_target_mask = y.notna()
X = X.loc[valid_target_mask]
y = y.loc[valid_target_mask].astype(int)

print(f"Target class distribution across all {NUM_CLASSES} classes:")
for cls_id in range(NUM_CLASSES):
    count = (y == cls_id).sum()
    print(f"  Class {cls_id:2d} ({IDX_TO_CLASS[cls_id]:26s}): {count:6d} samples ({count/len(y)*100:5.2f}%)")

# Stratified 75/25 Train-Test Split across all 10 classes
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Calculate balanced class sample weights to prevent majority normal class from overpowering
class_counts = y_train.value_counts().sort_index()
total_samples = len(y_train)
num_present_classes = len(class_counts)

class_weights = {
    class_id: total_samples / (num_present_classes * count)
    for class_id, count in class_counts.items()
}
sample_weights = y_train.map(class_weights).to_numpy()

# Initialize XGBoost 10-Class Classifier
xgb_clf = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    num_class=NUM_CLASSES,
    random_state=42,
    tree_method='hist',
    eval_metric='mlogloss',
    n_jobs=-1
)

print(f"\nTraining XGBoost Multi-Class Classifier on all {NUM_CLASSES} classes (including 'normal')...")
xgb_clf.fit(
    X_train, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_test, y_test)],
    verbose=False
)

# Predictions on Test Set
y_pred_xgb = xgb_clf.predict(X_test)
y_prob_xgb = xgb_clf.predict_proba(X_test)

macro_f1 = f1_score(y_test, y_pred_xgb, average='macro')
weighted_f1 = f1_score(y_test, y_pred_xgb, average='weighted')
roc_auc_multi = roc_auc_score(y_test, y_prob_xgb, multi_class='ovr', average='weighted')

print(f"\n=== 10-Class XGBoost Classifier Performance ===")
print(f"Macro F1-Score:      {macro_f1:.4f}")
print(f"Weighted F1-Score:   {weighted_f1:.4f}")
print(f"Multi-Class ROC-AUC: {roc_auc_multi:.4f}")

# Detailed 10-Class Classification Report
print("\nDetailed 10-Class Classification Report:")
print(classification_report(y_test, y_pred_xgb, labels=list(range(NUM_CLASSES)), target_names=ALL_CLASSES, digits=4))

## Step 7: Physics Consistency Layer

Implements a non-rigid, meteorological physics evidence layer:
- **Range Consistency**: Continuous soft-penalty on readings outside physical meteorological limits without rejecting legitimate weather extremes.
- **Rate-of-Change Consistency**: Temporal gradient checks on 1-hour deltas ($|\Delta T|, |\Delta RH|, |\Delta P|$).
- **Temperature–Humidity & Dew-Point Consistency**: Validates psychrometric relationships (Magnus dew point formula, super-saturation, and high-temp/high-humidity unphysical states).
- **Pressure Consistency**: Barometric deviations and rapid unphysical pressure jumps.
- **Cross-Sensor Consistency**: Validates physical coupling between temperature and humidity.

In [ ]:
# NEW: Step 7 Physics Consistency Layer Implementation

class PhysicsConsistencyEngine:
    """
    Meteorologically grounded Physics Consistency Layer.
    Computes continuous evidence scores rather than rigid binary drop filters,
    ensuring severe legitimate weather events are not erroneously discarded.
    """
    def __init__(self, config: Dict[str, Any] = PHYSICS_CONFIG):
        self.cfg = config

    def evaluate_dataframe(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Evaluates physics consistency across all telemetry records.
        Returns a DataFrame containing continuous evidence scores and violation indicators.
        """
        results = pd.DataFrame(index=df.index)

        # 1. Range Inconsistency Evidence (Soft-clipped degree of violation)
        t = df['temperature_c']
        rh = df['humidity_pct']
        p = df['pressure_hpa']

        t_range_viol = np.maximum(0, self.cfg['temp_min_c'] - t) / 10.0 + np.maximum(0, t - self.cfg['temp_max_c']) / 10.0
        rh_range_viol = np.maximum(0, self.cfg['humidity_min_pct'] - rh) / 20.0 + np.maximum(0, rh - self.cfg['humidity_max_pct']) / 20.0
        p_range_viol = np.maximum(0, self.cfg['pressure_min_hpa'] - p) / 25.0 + np.maximum(0, p - self.cfg['pressure_max_hpa']) / 25.0
        results['physics_range_score'] = np.clip((t_range_viol + rh_range_viol + p_range_viol) / 3.0, 0.0, 1.0)

        # 2. Rate of Change Inconsistency Evidence
        t_rate = df['temperature_c_diff_lag1'].abs() if 'temperature_c_diff_lag1' in df.columns else (t - t.shift(1)).abs().fillna(0)
        rh_rate = df['humidity_pct_diff_lag1'].abs() if 'humidity_pct_diff_lag1' in df.columns else (rh - rh.shift(1)).abs().fillna(0)
        p_rate = df['pressure_hpa_diff_lag1'].abs() if 'pressure_hpa_diff_lag1' in df.columns else (p - p.shift(1)).abs().fillna(0)

        t_rate_viol = np.maximum(0, t_rate - self.cfg['temp_max_rate_c_per_hr']) / self.cfg['temp_max_rate_c_per_hr']
        rh_rate_viol = np.maximum(0, rh_rate - self.cfg['humidity_max_rate_pct_per_hr']) / self.cfg['humidity_max_rate_pct_per_hr']
        p_rate_viol = np.maximum(0, p_rate - self.cfg['pressure_max_rate_hpa_per_hr']) / self.cfg['pressure_max_rate_hpa_per_hr']
        results['physics_rate_score'] = np.clip((t_rate_viol + rh_rate_viol + p_rate_viol) / 2.0, 0.0, 1.0)

        # 3. Psychrometric & Dew-Point Depression Inconsistency
        if 'dewpoint_c' in df.columns and 'dewpoint_depression_c' in df.columns:
            t_dew = df['dewpoint_c']
            dew_dep = df['dewpoint_depression_c']
        else:
            t_dew = calculate_dewpoint_magnus(t, rh)
            dew_dep = t - t_dew

        # Dew point cannot exceed dry-bulb temperature (allowing small measurement tolerance)
        dew_super_sat_viol = np.maximum(0, (t_dew - t) - 0.5) / 5.0
        # Extreme heat + Extreme humidity is physically impossible (wet bulb temperature limit ~35°C)
        extreme_wetbulb_viol = np.where((t > 40.0) & (rh > 85.0), ((t - 40.0) / 10.0) * ((rh - 85.0) / 15.0), 0.0)
        results['physics_dewpoint_score'] = np.clip(dew_super_sat_viol + extreme_wetbulb_viol, 0.0, 1.0)

        # 4. Cross-Sensor & Multivariate Inconsistency
        cross_viol = np.where((t > 42.0) & (rh > 90.0), 0.9, 0.0)
        if 'temperature_c_frozen_count' in df.columns:
            frozen_viol = np.where(df['temperature_c_frozen_count'] >= 6, np.clip(df['temperature_c_frozen_count'] / 15.0, 0.0, 1.0), 0.0)
        else:
            frozen_viol = 0.0
        results['physics_cross_score'] = np.clip(cross_viol + frozen_viol, 0.0, 1.0)

        # 5. Composite Physics Consistency Evidence Score
        results['physics_evidence_score'] = (
            self.cfg['weight_range'] * results['physics_range_score'] +
            self.cfg['weight_rate'] * results['physics_rate_score'] +
            self.cfg['weight_dewpoint'] * results['physics_dewpoint_score'] +
            self.cfg['weight_cross_sensor'] * results['physics_cross_score']
        ).clip(0.0, 1.0)

        return results


# Instantiate and compute physics evidence on df_injected
physics_engine = PhysicsConsistencyEngine(PHYSICS_CONFIG)
df_physics_evidence = physics_engine.evaluate_dataframe(df_injected)

# Merge physics evidence into df_injected
for col in df_physics_evidence.columns:
    df_injected[col] = df_physics_evidence[col]

print("Physics Consistency Evidence computed successfully:")
print(df_injected[['physics_range_score', 'physics_rate_score', 'physics_dewpoint_score', 'physics_cross_score', 'physics_evidence_score']].describe().round(4))

## Step 8: Multi-Source Evidence Fusion Layer

Integrates all 5 independent evidence streams into a unified calibrated anomaly evidence score:
1. **Isolation Forest Novelty** ($E_{\text{novelty}}$): Continuous unsupervised outlier score.
2. **Temporal / Statistical Evidence** ($E_{\text{temporal}}$): Rolling 24-hour z-scores, frozen counts, and local rate deltas.
3. **Spatial Consensus Evidence** ($E_{\text{spatial}}$): Regional cluster mean deviation and spatial z-score.
4. **Physics Consistency Evidence** ($E_{\text{physics}}$): Non-rigid physical and psychrometric rule evaluations.
5. **Classifier Evidence** ($E_{\text{xgb}}$): Multi-class anomaly confidence ($1.0 - P(\text{normal})$).

In [ ]:
# NEW: Step 8 Multi-Source Evidence Fusion Layer Implementation

class EvidenceFusionEngine:
    """
    Multi-Source Evidence Fusion Engine.
    Fuses 5 distinct evidence channels into a unified, calibrated anomaly score.
    """
    def __init__(self, weights: Dict[str, float] = FUSION_CONFIG):
        self.weights = weights
        total_w = sum(weights.values())
        self.norm_weights = {k: v / total_w for k, v in weights.items()}

    def fuse(self,
             iforest_novelty: np.ndarray,
             temporal_evidence: np.ndarray,
             spatial_evidence: np.ndarray,
             physics_evidence: np.ndarray,
             xgb_anomaly_prob: np.ndarray) -> np.ndarray:
        """
        Calculates normalized weighted fused evidence score.
        """
        w = self.norm_weights
        fused = (
            w['w_iforest'] * np.clip(iforest_novelty, 0, 1) +
            w['w_temporal'] * np.clip(temporal_evidence, 0, 1) +
            w['w_spatial'] * np.clip(spatial_evidence, 0, 1) +
            w['w_physics'] * np.clip(physics_evidence, 0, 1) +
            w['w_xgboost'] * np.clip(xgb_anomaly_prob, 0, 1)
        )
        return np.clip(fused, 0.0, 1.0)


# Extract individual normalized evidence channels on df_injected
e_iforest = df_injected['iforest_ml_score'].to_numpy()

# Temporal Evidence: combination of 24h rolling z-scores and frozen counts
temp_z = df_injected['temperature_c_roll24_zscore'].to_numpy() / 3.5
hum_z = df_injected['humidity_pct_roll24_zscore'].to_numpy() / 3.5
press_z = df_injected['pressure_hpa_roll24_zscore'].to_numpy() / 3.5
frozen_pen = (df_injected['temperature_c_frozen_count'].to_numpy() >= 5).astype(float) * 0.8
e_temporal = np.clip(np.maximum.reduce([temp_z, hum_z, press_z, frozen_pen]), 0.0, 1.0)

# Spatial Evidence: regional cluster deviation z-scores
e_spatial = np.clip(df_injected['spatial_temp_zscore'].abs().to_numpy() / 3.0, 0.0, 1.0)

# Physics Evidence from Step 7
e_physics = df_injected['physics_evidence_score'].to_numpy()

# XGBoost Evidence: (1.0 - P(normal))
all_probs_xgb = xgb_clf.predict_proba(df_injected_encoded[feature_cols])
e_xgb_anomaly = 1.0 - all_probs_xgb[:, CLASS_TO_IDX['normal']]

# Execute Evidence Fusion
fusion_engine = EvidenceFusionEngine(FUSION_CONFIG)
fused_scores = fusion_engine.fuse(
    iforest_novelty=e_iforest,
    temporal_evidence=e_temporal,
    spatial_evidence=e_spatial,
    physics_evidence=e_physics,
    xgb_anomaly_prob=e_xgb_anomaly
)

df_injected['fused_anomaly_score'] = fused_scores
df_injected['temporal_evidence'] = e_temporal
df_injected['spatial_evidence'] = e_spatial
df_injected['xgb_anomaly_evidence'] = e_xgb_anomaly

print("Multi-Source Evidence Fusion complete:")
print(f"Fused Score Distribution: Min={fused_scores.min():.3f}, Mean={fused_scores.mean():.3f}, Max={fused_scores.max():.3f}")

## Step 9: Decision Engine (Normal / Known Anomaly / Novel Anomaly)

Outputs 3 distinct triage states:
- `normal`: Telemetry operates within nominal parameters (low fused score, high $P(\text{normal})$).
- `known_anomaly`: High fused anomaly evidence and confident match to one of the 9 known physical failure modes.
- `novel_anomaly`: High anomaly/novelty evidence with low confidence across known classes (or unmodeled failure pattern), avoiding false forced classification.

In [ ]:
# NEW: Step 9 Decision Engine Implementation

class DecisionEngine:
    """
    3-Way Decision Engine:
    Triages instances into 'normal', 'known_anomaly', or 'novel_anomaly'.
    """
    def __init__(self, config: Dict[str, Any] = DECISION_CONFIG):
        self.anomaly_thresh = config['anomaly_threshold']
        self.known_thresh = config['known_class_threshold']
        self.novelty_thresh = config['novelty_threshold']

    def decide(self,
               fused_score: float,
               xgb_probs: np.ndarray,
               iforest_score: float) -> Dict[str, Any]:
        """
        Evaluates single observation and assigns decision state, root cause, and confidence.
        """
        # Exclude normal (index 0) to evaluate known anomaly candidate classes
        anomaly_probs = xgb_probs[1:]
        top_anomaly_idx = int(np.argmax(anomaly_probs) + 1)
        top_anomaly_prob = float(anomaly_probs[top_anomaly_idx - 1])
        normal_prob = float(xgb_probs[0])

        is_flagged_anomaly = (fused_score >= self.anomaly_thresh) or (normal_prob < 0.50)

        if not is_flagged_anomaly:
            return {
                "decision": "normal",
                "root_cause": "normal",
                "confidence": normal_prob,
                "is_anomaly": 0
            }
        else:
            # Check if it confidently matches a known root cause
            if top_anomaly_prob >= self.known_thresh:
                return {
                    "decision": "known_anomaly",
                    "root_cause": IDX_TO_CLASS[top_anomaly_idx],
                    "confidence": top_anomaly_prob,
                    "is_anomaly": 1
                }
            else:
                # High anomaly evidence but low known class confidence => Novel Anomaly
                return {
                    "decision": "novel_anomaly",
                    "root_cause": "novel_anomaly",
                    "confidence": float(max(fused_score, iforest_score)),
                    "is_anomaly": 1
                }

    def evaluate_batch(self,
                       fused_scores: np.ndarray,
                       xgb_probs: np.ndarray,
                       iforest_scores: np.ndarray) -> pd.DataFrame:
        results = []
        for i in range(len(fused_scores)):
            res = self.decide(fused_scores[i], xgb_probs[i], iforest_scores[i])
            results.append(res)
        return pd.DataFrame(results)


# Execute Decision Engine on df_injected
decision_engine = DecisionEngine(DECISION_CONFIG)
df_decisions = decision_engine.evaluate_batch(
    fused_scores=df_injected['fused_anomaly_score'].to_numpy(),
    xgb_probs=all_probs_xgb,
    iforest_scores=df_injected['iforest_ml_score'].to_numpy()
)

for col in df_decisions.columns:
    df_injected[col] = df_decisions[col]

print("Decision Engine Triaged Output Summary:")
print(df_injected['decision'].value_counts())
print("\nRoot Cause Diagnoses Breakdown:")
print(df_injected['root_cause'].value_counts())

## Step 10: Multi-Factor Severity Engine (LOW / MEDIUM / HIGH / CRITICAL)

Computes operational alert severity considering anomaly strength, temporal persistence, multi-sensor coupling, physics violations, and model confidence.

In [ ]:
# NEW: Step 10 Severity Engine Implementation

class SeverityEngine:
    """
    Configurable Multi-Factor Severity Engine.
    Ranks anomalies into LOW, MEDIUM, HIGH, or CRITICAL tiers.
    """
    def __init__(self, config: Dict[str, Any] = SEVERITY_CONFIG):
        self.cfg = config
        self.thresholds = config['thresholds']

    def calculate_severity(self,
                           decision: str,
                           fused_score: float,
                           confidence: float,
                           frozen_count: int,
                           physics_score: float,
                           root_cause: str) -> Dict[str, Any]:
        if decision == 'normal':
            return {"severity": "NONE", "severity_score": 0.0}

        # Factor 1: Anomaly strength
        strength = float(fused_score)

        # Factor 2: Temporal persistence
        persistence = np.clip(frozen_count / 12.0, 0.0, 1.0)

        # Factor 3: Model confidence
        conf = float(confidence)

        # Factor 4: Multi-sensor involvement & critical root causes
        critical_causes = {'multivariate_inconsistency', 'missing_data', 'freeze'}
        multi_sensor = 0.9 if root_cause in critical_causes else float(physics_score)

        # Composite severity score
        score = (
            self.cfg['weight_anomaly_strength'] * strength +
            self.cfg['weight_persistence'] * persistence +
            self.cfg['weight_confidence'] * conf +
            self.cfg['weight_multi_sensor'] * multi_sensor
        )
        score = float(np.clip(score, 0.0, 1.0))

        # Map to operational tiers
        if score >= self.thresholds['CRITICAL']:
            tier = "CRITICAL"
        elif score >= self.thresholds['HIGH']:
            tier = "HIGH"
        elif score >= self.thresholds['MEDIUM']:
            tier = "MEDIUM"
        else:
            tier = "LOW"

        return {"severity": tier, "severity_score": round(score, 4)}

    def evaluate_dataframe(self, df: pd.DataFrame) -> pd.DataFrame:
        results = []
        for _, row in df.iterrows():
            frozen = int(row.get('temperature_c_frozen_count', 1))
            res = self.calculate_severity(
                decision=row['decision'],
                fused_score=row['fused_anomaly_score'],
                confidence=row['confidence'],
                frozen_count=frozen,
                physics_score=row.get('physics_evidence_score', 0.0),
                root_cause=row['root_cause']
            )
            results.append(res)
        return pd.DataFrame(results, index=df.index)


# Execute Severity Engine
severity_engine = SeverityEngine(SEVERITY_CONFIG)
df_severity = severity_engine.evaluate_dataframe(df_injected)
df_injected['severity'] = df_severity['severity']
df_injected['severity_score'] = df_severity['severity_score']

print("Severity Distribution across All Records:")
print(df_injected['severity'].value_counts())

## Step 11: Transparent 0–100 Sensor Health Scoring Engine

Provides an explainable, transparent 0–100 health index for each station and sensor with itemized deduction breakdowns.

In [ ]:
# NEW: Step 11 Sensor Health Scoring Engine Implementation

class SensorHealthScorer:
    """
    Transparent 0–100 Sensor Health Scoring Engine.
    Calculates health index with explicit itemized penalty deductions.
    """
    def __init__(self, config: Dict[str, Any] = HEALTH_CONFIG):
        self.cfg = config

    def calculate_health(self,
                         decision: str,
                         severity_tier: str,
                         root_cause: str,
                         physics_score: float,
                         spatial_dev_score: float,
                         is_nan_dropout: bool = False) -> Dict[str, Any]:
        base = self.cfg['base_score']
        deductions = {
            "anomaly_severity_penalty": 0.0,
            "drift_offset_penalty": 0.0,
            "missing_dropout_penalty": 0.0,
            "physics_inconsistency_penalty": 0.0
        }

        if decision != 'normal':
            # 1. Anomaly severity penalty
            sev_map = {"LOW": 10.0, "MEDIUM": 20.0, "HIGH": 30.0, "CRITICAL": 35.0, "NONE": 0.0}
            deductions["anomaly_severity_penalty"] = sev_map.get(severity_tier, 15.0)

            # 2. Drift / Offset degradation penalty
            if root_cause in ['drift', 'offset']:
                deductions["drift_offset_penalty"] = 20.0

            # 3. Missing data / dropout penalty
            if root_cause == 'missing_data' or is_nan_dropout:
                deductions["missing_dropout_penalty"] = 20.0

            # 4. Physics / Spatial inconsistency penalty
            phys_pen = min(self.cfg['max_physics_deduction'], physics_score * 20.0 + spatial_dev_score * 10.0)
            deductions["physics_inconsistency_penalty"] = round(phys_pen, 1)

        total_deduction = sum(deductions.values())
        final_score = int(np.clip(base - total_deduction, 0.0, 100.0))

        return {
            "health_score": final_score,
            "deductions": deductions,
            "status": "HEALTHY" if final_score >= 85 else ("DEGRADED" if final_score >= 50 else "CRITICAL")
        }

    def evaluate_dataframe(self, df: pd.DataFrame) -> pd.DataFrame:
        results = []
        for _, row in df.iterrows():
            res = self.calculate_health(
                decision=row['decision'],
                severity_tier=row['severity'],
                root_cause=row['root_cause'],
                physics_score=row.get('physics_evidence_score', 0.0),
                spatial_dev_score=row.get('neighbor_dev_score', 0.0),
                is_nan_dropout=(row['root_cause'] == 'missing_data')
            )
            results.append({
                "health_score": res['health_score'],
                "health_status": res['status']
            })
        return pd.DataFrame(results, index=df.index)


# Execute Sensor Health Scoring
health_scorer = SensorHealthScorer(HEALTH_CONFIG)
df_health = health_scorer.evaluate_dataframe(df_injected)
df_injected['health_score'] = df_health['health_score']
df_injected['health_status'] = df_health['health_status']

print("Sensor Health Score Summary:")
print(df_injected['health_score'].describe())
print("\nHealth Status Breakdown:")
print(df_injected['health_status'].value_counts())

## Step 12: Human-Readable SHAP Explainability & Physical Attribution

Transforms quantitative SHAP attributions into natural human-readable diagnostic evidence sentences explaining why each alert was generated.

In [ ]:
# MODIFIED & NEW: Step 12 Enhanced Human-Readable SHAP Diagnostic Attribution

# Initialize TreeExplainer for 10-Class XGBoost
explainer = shap.TreeExplainer(xgb_clf)

# Sample representative test set for SHAP computation
np.random.seed(42)
sample_size = min(1500, len(X_test))
sample_idx = np.random.choice(X_test.index, size=sample_size, replace=False)
X_shap_sample = X_test.loc[sample_idx]
y_shap_sample = y_test.loc[sample_idx]

print(f"Computing SHAP values on {sample_size} test observations across all {NUM_CLASSES} classes...")
shap_values = explainer(X_shap_sample)


# NEW: Helper function to generate human-readable physical evidence statements from SHAP attributions
def explain_instance_shap_human_readable(row_features: pd.DataFrame, target_class_id: int, top_k: int = 4) -> List[Dict[str, Any]]:
    """
    Extracts top contributing SHAP features and translates them into natural human-readable physical evidence statements.
    """
    inst_shap = explainer(row_features)
    if len(inst_shap.values.shape) == 3:
        class_shap_vals = inst_shap.values[0, :, target_class_id]
    else:
        class_shap_vals = inst_shap.values[0, :]

    top_indices = np.argsort(class_shap_vals)[-top_k:][::-1]
    explanations = []

    domain_descriptions = {
        'spatial_temp_diff': "Regional temperature deviation from cluster stations",
        'spatial_temp_zscore': "Spatial temperature anomaly z-score",
        'temperature_c_roll24_zscore': "24-hour temperature baseline statistical deviation",
        'temperature_c_diff_lag1': "1-hour rapid temperature rate of change",
        'temperature_c_frozen_count': "Consecutive unvarying temperature readings (stuck sensor)",
        'humidity_pct_roll24_zscore': "24-hour humidity baseline statistical deviation",
        'pressure_hpa_diff_lag1': "1-hour sudden barometric pressure jump",
        'dewpoint_depression_c': "Psychrometric dew-point depression inconsistency",
        'temp_hum_ratio': "Anomalous temperature-to-humidity physical coupling ratio",
        'station_neighbor_consistency': "Inter-station consensus consistency metric"
    }

    for idx in top_indices:
        feat_name = feature_cols[idx]
        val = float(row_features.iloc[0, idx])
        shap_val = float(class_shap_vals[idx])
        desc = domain_descriptions.get(feat_name, f"Physical telemetry feature '{feat_name}'")
        statement = f"{desc} (value = {val:.2f}) increased anomaly confidence by +{shap_val:.3f}"
        explanations.append({
            "feature": feat_name,
            "value": val,
            "shap_contribution": shap_val,
            "human_readable_statement": statement
        })
    return explanations

## Step 13: Maintenance Recommendation Engine

Maps detected root causes and operational severity tiers to domain-specific, actionable field maintenance and calibration protocols.

In [ ]:
# NEW: Step 13 Maintenance Recommendation Engine Implementation

class MaintenanceRecommendationEngine:
    """
    Maintenance Recommendation Engine.
    Maps detected failure mode and severity into precise physical engineering actions.
    """
    ACTION_CATALOG = {
        "temperature_spike": {
            "action": "Inspect RTD / thermistor probe wiring for loose grounding; inspect solar radiation shield for aspirated fan failure or solar heat entrapment.",
            "priority": "High if persisting; check ADC board channel."
        },
        "humidity_spike": {
            "action": "Inspect capacitive hygrometer polymer element for liquid water condensation or membrane contamination; clean or replace protective sintered cap.",
            "priority": "Medium; check sensor enclosure seal."
        },
        "pressure_jump": {
            "action": "Inspect barometric port venting tube for insect/dust blockage; verify static pressure port against calibrated digital barometer standard.",
            "priority": "High; verify manifold pressure seal."
        },
        "freeze": {
            "action": "Sensor hardware unresponsive / stuck value. Perform remote bus power-cycle (I2C/RS485); if unrecovered, dispatch technician to replace sensor transducer unit.",
            "priority": "Critical; immediate hardware swap required."
        },
        "drift": {
            "action": "Progressive calibration degradation detected. Schedule two-point calibration check against field reference probe; apply span/zero offset compensation in station logger.",
            "priority": "Medium; schedule during next routine maintenance window."
        },
        "offset": {
            "action": "Static baseline calibration offset detected. Perform zero-calibration or update calibration intercept in data logger firmware.",
            "priority": "Low to Medium; recalibrate offset parameter."
        },
        "missing_data": {
            "action": "Telemetry dropout / transmission loss. Check solar PV panel charge controller, battery terminal voltage, LoRa/GSM antenna signal, and data logger serial connection.",
            "priority": "High; inspect power supply and cellular modem."
        },
        "multivariate_inconsistency": {
            "action": "Thermodynamic/psychrometric inconsistency between T and RH sensors. Perform simultaneous inspection of both temperature and humidity sensing elements; check for moisture ingress.",
            "priority": "High; dual-sensor calibration audit."
        },
        "spatial_inconsistency": {
            "action": "Station readings diverge significantly from regional cluster consensus. Cross-examine neighbor stations in cluster; inspect for local micro-climate obstruction or micro-siting interference.",
            "priority": "Medium; regional consensus verification."
        },
        "novel_anomaly": {
            "action": "Unclassified novel anomaly pattern detected. Dispatch telemetry engineer for comprehensive multi-sensor diagnostic sweep, firmware signal log extraction, and physical site inspection.",
            "priority": "High; requires engineering investigation."
        },
        "normal": {
            "action": "All sensors operating within nominal meteorological and physical tolerances. Routine scheduled maintenance cycle.",
            "priority": "Nominal; no action required."
        }
    }

    @classmethod
    def recommend(cls, root_cause: str, severity: str) -> Dict[str, str]:
        info = cls.ACTION_CATALOG.get(root_cause, cls.ACTION_CATALOG["normal"])
        return {
            "recommended_action": info["action"],
            "engineering_priority": f"{severity} - {info['priority']}"
        }

## Step 14: Structured LLM-Ready Diagnostic Report & LLM Reporting Layer

Constructs a complete, machine-readable JSON diagnostic payload and human/LLM-ready markdown incident briefing without requiring external APIs.

In [ ]:
# NEW: Step 14 Structured LLM Diagnostic Report Object & LLM Reporting Layer

def generate_diagnostic_report(row_index: int,
                               df_source: pd.DataFrame,
                               df_enc: pd.DataFrame) -> Dict[str, Any]:
    """
    Generates a comprehensive, self-contained LLM-ready diagnostic report dictionary.
    """
    row_raw = df_source.loc[row_index]
    row_feat = df_enc.loc[[row_index], feature_cols]

    # Predictions & Decision
    pred_probs = xgb_clf.predict_proba(row_feat)[0]
    fused_score = float(row_raw.get('fused_anomaly_score', 0.0))
    iforest_score = float(row_raw.get('iforest_ml_score', 0.0))

    decision_info = decision_engine.decide(fused_score, pred_probs, iforest_score)
    decision = decision_info['decision']
    root_cause = decision_info['root_cause']
    confidence = decision_info['confidence']

    # Severity & Health
    frozen = int(row_raw.get('temperature_c_frozen_count', 1))
    phys_score = float(row_raw.get('physics_evidence_score', 0.0))
    sev_info = severity_engine.calculate_severity(
        decision, fused_score, confidence, frozen, phys_score, root_cause
    )
    health_info = health_scorer.calculate_health(
        decision, sev_info['severity'], root_cause, phys_score,
        float(row_raw.get('neighbor_dev_score', 0.0))
    )

    # Maintenance recommendation
    maint = MaintenanceRecommendationEngine.recommend(root_cause, sev_info['severity'])

    # Human readable SHAP drivers
    cls_id = CLASS_TO_IDX.get(root_cause, 0)
    shap_factors = explain_instance_shap_human_readable(row_feat, target_class_id=cls_id, top_k=4)

    report = {
        "incident_metadata": {
            "station_id": str(row_raw.get('station_id', 'N/A')),
            "station_name": str(row_raw.get('station_name', 'N/A')),
            "city": str(row_raw.get('city', 'N/A')),
            "timestamp": str(row_raw.get('timestamp', 'N/A')),
            "cluster": str(row_raw.get('cluster', 'N/A'))
        },
        "telemetry_readings": {
            "temperature_c": float(row_raw.get('temperature_c', np.nan)),
            "humidity_pct": float(row_raw.get('humidity_pct', np.nan)),
            "pressure_hpa": float(row_raw.get('pressure_hpa', np.nan)),
            "dewpoint_c": float(row_raw.get('dewpoint_c', np.nan))
        },
        "diagnosis": {
            "decision": decision,
            "root_cause": root_cause,
            "confidence_pct": round(confidence * 100, 2),
            "true_ground_truth": str(row_raw.get('root_cause', 'Unknown')),
            "severity": sev_info['severity'],
            "severity_score": sev_info['severity_score'],
            "sensor_health_score": health_info['health_score'],
            "health_status": health_info['status'],
            "itemized_health_deductions": health_info['deductions']
        },
        "multi_source_evidence": {
            "fused_anomaly_score": round(fused_score, 4),
            "iforest_novelty_score": round(iforest_score, 4),
            "temporal_evidence_score": round(float(row_raw.get('temporal_evidence', 0.0)), 4),
            "spatial_evidence_score": round(float(row_raw.get('spatial_evidence', 0.0)), 4),
            "physics_evidence_score": round(phys_score, 4),
            "xgb_anomaly_probability": round(float(row_raw.get('xgb_anomaly_evidence', 0.0)), 4)
        },
        "physics_checks": {
            "range_score": round(float(row_raw.get('physics_range_score', 0.0)), 3),
            "rate_score": round(float(row_raw.get('physics_rate_score', 0.0)), 3),
            "dewpoint_score": round(float(row_raw.get('physics_dewpoint_score', 0.0)), 3),
            "cross_sensor_score": round(float(row_raw.get('physics_cross_score', 0.0)), 3)
        },
        "shap_physical_attributions": shap_factors,
        "maintenance": maint
    }
    return report


def format_llm_report(report: Dict[str, Any]) -> str:
    """
    Formats structured diagnostic report into an executive human/LLM-readable briefing text.
    """
    meta = report['incident_metadata']
    readings = report['telemetry_readings']
    diag = report['diagnosis']
    ev = report['multi_source_evidence']
    maint = report['maintenance']

    lines = [
        "=" * 75,
        "  SKYGUARD AI — TELEMETRY INCIDENT & DIAGNOSTIC BRIEFING",
        "=" * 75,
        f"Station:       {meta['station_name']} (ID: {meta['station_id']}, Cluster: {meta['cluster']})",
        f"Timestamp:     {meta['timestamp']}",
        f"Readings:      Temp = {readings['temperature_c']:.1f}°C | Hum = {readings['humidity_pct']:.1f}% | Press = {readings['pressure_hpa']:.1f} hPa | DewPoint = {readings['dewpoint_c']:.1f}°C",
        "-" * 75,
        f"TRIAGE STATE:  {diag['decision'].upper()}",
        f"ROOT CAUSE:    {diag['root_cause'].upper()}",
        f"CONFIDENCE:    {diag['confidence_pct']:.1f}%",
        f"SEVERITY:      {diag['severity']} (Score: {diag['severity_score']})",
        f"SENSOR HEALTH: {diag['sensor_health_score']}/100 ({diag['health_status']})",
        "-" * 75,
        "EVIDENCE BREAKDOWN:",
        f"  • Fused Anomaly Score:    {ev['fused_anomaly_score']:.3f}",
        f"  • Isolation Forest Score: {ev['iforest_novelty_score']:.3f}",
        f"  • Physics Evidence Score: {ev['physics_evidence_score']:.3f}",
        f"  • Spatial Deviation:      {ev['spatial_evidence_score']:.3f}",
        f"  • Temporal & Persistence: {ev['temporal_evidence_score']:.3f}",
        "-" * 75,
        "TOP PHYSICAL ATTRIBUTION DRIVERS (SHAP Explainability):"
    ]
    for i, factor in enumerate(report['shap_physical_attributions'], 1):
        lines.append(f"  {i}. {factor['human_readable_statement']}")
    lines.extend([
        "-" * 75,
        "RECOMMENDED MAINTENANCE PROTOCOL:",
        f"  Action:   {maint['recommended_action']}",
        f"  Priority: {maint['engineering_priority']}",
        "=" * 75
    ])
    return "\n".join(lines)

## Step 15: Operator Feedback Logging & Safe Model Improvement Workflow

Enables field operators to validate or correct alerts (`confirmed`, `false_alarm`, `corrected_root_cause`, `operator_comment`) and provides a safe, quarantined workflow for incremental model retraining.

In [ ]:
# NEW: Step 15 Operator Feedback Interface & Safe Model Improvement Workflow

class OperatorFeedbackStore:
    """
    Operator Feedback Data Store.
    Captures field technician reviews and maintains an auditable feedback log.
    """
    def __init__(self):
        self.feedback_records: List[Dict[str, Any]] = []

    def log_feedback(self,
                     incident_id: str,
                     station_id: str,
                     timestamp: str,
                     predicted_root_cause: str,
                     predicted_severity: str,
                     confirmed: bool,
                     false_alarm: bool,
                     corrected_root_cause: Optional[str] = None,
                     operator_comment: str = "",
                     operator_id: str = "OP_TECH_01") -> Dict[str, Any]:
        record = {
            "incident_id": incident_id,
            "station_id": station_id,
            "timestamp": timestamp,
            "predicted_root_cause": predicted_root_cause,
            "predicted_severity": predicted_severity,
            "confirmed": confirmed,
            "false_alarm": false_alarm,
            "corrected_root_cause": corrected_root_cause if (false_alarm and corrected_root_cause) else (predicted_root_cause if confirmed else "unvalidated"),
            "operator_comment": operator_comment,
            "operator_id": operator_id,
            "review_timestamp": datetime.now().isoformat(),
            "validation_status": "VERIFIED_GOLDEN" if (confirmed or corrected_root_cause) else "QUARANTINED"
        }
        self.feedback_records.append(record)
        return record

    def get_validated_training_data(self) -> pd.DataFrame:
        """
        Safeguard: Returns strictly human-verified records for model retraining.
        """
        df_fb = pd.DataFrame(self.feedback_records)
        if df_fb.empty:
            return df_fb
        return df_fb[df_fb['validation_status'] == 'VERIFIED_GOLDEN'].copy()


# Demonstration of Operator Feedback Logging & Safe Retraining Pipeline
feedback_store = OperatorFeedbackStore()

# Sample operator interaction logs
feedback_store.log_feedback(
    incident_id="INC-2026-0801",
    station_id="STATION_004",
    timestamp="2026-08-20 14:00:00",
    predicted_root_cause="temperature_spike",
    predicted_severity="HIGH",
    confirmed=True,
    false_alarm=False,
    operator_comment="Loose RTD terminal wire confirmed in field inspection."
)

feedback_store.log_feedback(
    incident_id="INC-2026-0802",
    station_id="STATION_012",
    timestamp="2026-08-20 16:00:00",
    predicted_root_cause="offset",
    predicted_severity="MEDIUM",
    confirmed=False,
    false_alarm=True,
    corrected_root_cause="drift",
    operator_comment="Sensor was drifting gradually rather than constant offset."
)

print("Sample Operator Feedback Store Records:")
df_fb_demo = pd.DataFrame(feedback_store.feedback_records)
print(df_fb_demo[['incident_id', 'station_id', 'predicted_root_cause', 'corrected_root_cause', 'validation_status', 'operator_comment']])

## Step 16: Unified End-to-End Prediction Pipeline & Comprehensive Architecture Evaluation

Evaluates the complete target architecture on test telemetry, including 10-class metrics, novel anomaly detection performance, severity distributions, and sensor health metrics.

In [ ]:
# NEW: Step 16 End-to-End Pipeline Execution & Comprehensive Evaluation

def run_skyguard_pipeline(df_raw_input: pd.DataFrame, df_encoded_input: pd.DataFrame) -> pd.DataFrame:
    """
    Executes the entire SkyGuard multi-tier pipeline:
    Telemetry -> Physics Consistency -> Novelty & Classification -> Evidence Fusion -> Decision -> Severity -> Health -> Maintenance.
    """
    df_out = df_raw_input.copy().reset_index(drop=True)

    # 1. Physics consistency
    df_phys = physics_engine.evaluate_dataframe(df_out)
    for c in df_phys.columns:
        df_out[c] = df_phys[c]

    # 2. Isolation forest novelty
    raw_if = iso_model.decision_function(df_encoded_input[feature_cols])
    if_scores = 1.0 - ((raw_if - raw_if.min()) / (raw_if.max() - raw_if.min() + 1e-9))
    df_out['iforest_ml_score'] = if_scores

    # 3. XGBoost probabilities
    xgb_probs = xgb_clf.predict_proba(df_encoded_input[feature_cols])
    xgb_anom_prob = 1.0 - xgb_probs[:, CLASS_TO_IDX['normal']]
    df_out['xgb_anomaly_evidence'] = xgb_anom_prob

    # 4. Temporal and spatial evidence
    temp_z = df_out['temperature_c_roll24_zscore'].to_numpy() / 3.5
    frozen_pen = (df_out['temperature_c_frozen_count'].to_numpy() >= 5).astype(float) * 0.8
    df_out['temporal_evidence'] = np.clip(np.maximum(temp_z, frozen_pen), 0.0, 1.0)
    df_out['spatial_evidence'] = np.clip(df_out['spatial_temp_zscore'].abs().to_numpy() / 3.0, 0.0, 1.0)

    # 5. Evidence Fusion
    df_out['fused_anomaly_score'] = fusion_engine.fuse(
        iforest_novelty=if_scores,
        temporal_evidence=df_out['temporal_evidence'].to_numpy(),
        spatial_evidence=df_out['spatial_evidence'].to_numpy(),
        physics_evidence=df_out['physics_evidence_score'].to_numpy(),
        xgb_anomaly_prob=xgb_anom_prob
    )

    # 6. Decision Engine
    df_dec = decision_engine.evaluate_batch(
        fused_scores=df_out['fused_anomaly_score'].to_numpy(),
        xgb_probs=xgb_probs,
        iforest_scores=if_scores
    )
    for c in df_dec.columns:
        df_out[c] = df_dec[c]

    # 7. Severity Engine
    df_sev = severity_engine.evaluate_dataframe(df_out)
    df_out['severity'] = df_sev['severity']
    df_out['severity_score'] = df_sev['severity_score']

    # 8. Sensor Health Score
    df_h = health_scorer.evaluate_dataframe(df_out)
    df_out['health_score'] = df_h['health_score']
    df_out['health_status'] = df_h['health_status']

    return df_out


# Run comprehensive evaluation on test set
df_test_raw = df_injected.loc[test_idx].reset_index(drop=True)
df_test_enc = df_injected_encoded.loc[test_idx].reset_index(drop=True)
test_pipeline_results = run_skyguard_pipeline(df_test_raw, df_test_enc)

# 1. Anomaly Detection Binary Metrics
y_true_test = test_pipeline_results['anomaly_label'].to_numpy()
y_pred_test = test_pipeline_results['is_anomaly'].to_numpy()
fused_test_scores = test_pipeline_results['fused_anomaly_score'].to_numpy()

print("=" * 70)
print("  SKYGUARD UPGRADED ARCHITECTURE EVALUATION ON TEST TELEMETRY")
print("=" * 70)
print(f"Fused Anomaly ROC-AUC: {roc_auc_score(y_true_test, fused_test_scores):.4f}")
print(f"Fused Anomaly PR-AUC:  {average_precision_score(y_true_test, fused_test_scores):.4f}")
print(f"Anomaly Precision:    {precision_score(y_true_test, y_pred_test):.4f}")
print(f"Anomaly Recall:       {recall_score(y_true_test, y_pred_test):.4f}")
print(f"Anomaly F1-Score:     {f1_score(y_true_test, y_pred_test):.4f}")

# 2. Novel Anomaly Detection Evaluation
print("\n--- Novel / Unseen Anomaly Detection Performance ---")
df_novel_test = inject_novel_anomalies(df_test_raw, random_state=101, num_instances=35)
df_novel_enc = df_test_enc.copy()
novel_eval_results = run_skyguard_pipeline(df_novel_test, df_novel_enc)
novel_instances = novel_eval_results[novel_eval_results['root_cause'] == 'novel_anomaly']
novel_detection_rate = (novel_instances['is_anomaly'] == 1).mean() * 100
novel_flag_as_novel_pct = (novel_instances['decision'] == 'novel_anomaly').mean() * 100
print(f"Novel Anomaly Overall Detection Rate:   {novel_detection_rate:.1f}%")
print(f"Identified specifically as Novel Anomaly: {novel_flag_as_novel_pct:.1f}% (Rejected from forced known classes)")

# 3. Severity and Health Visualizations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
test_pipeline_results['severity'].value_counts().plot(kind='bar', color=['#2ecc71', '#f39c12', '#e67e22', '#e74c3c'], ax=axes[0])
axes[0].set_title('Operational Severity Tier Distribution')
axes[0].set_ylabel('Count')

sns.histplot(test_pipeline_results['health_score'], bins=20, kde=True, color='#3498db', ax=axes[1])
axes[1].set_title('0–100 Sensor Health Score Distribution')
axes[1].set_xlabel('Sensor Health Score (0 = Failed, 100 = Nominal)')
plt.tight_layout()
plt.show()

## Step 17: Comprehensive Demonstration Cases (Normal, Known Anomaly, Novel Anomaly)

Demonstrates the full end-to-end diagnosis, human-readable SHAP explanation, structured LLM diagnostic report, and maintenance action for all 3 operational cases.

In [ ]:
# NEW: Step 17 Multi-Case Demonstration Outputs (Normal, Known Anomaly, Novel Anomaly)

print("=" * 80)
print(" CASE 1: DEMONSTRATION OF NOMINAL / NORMAL TELEMETRY INSTANCE")
print("=" * 80)
normal_matches = df_injected[df_injected['root_cause'] == 'normal'].index
if len(normal_matches) > 0:
    demo_idx_normal = normal_matches[0]
    report_normal = generate_diagnostic_report(demo_idx_normal, df_injected, df_injected_encoded)
    print(format_llm_report(report_normal))

print("\n" + "=" * 80)
print(" CASE 2: DEMONSTRATION OF KNOWN ROOT CAUSE ANOMALY (Temperature Spike / Freeze)")
print("=" * 80)
known_matches = df_injected[df_injected['root_cause'] == 'temperature_spike'].index
if len(known_matches) > 0:
    demo_idx_known = known_matches[0]
    report_known = generate_diagnostic_report(demo_idx_known, df_injected, df_injected_encoded)
    print(format_llm_report(report_known))

print("\n" + "=" * 80)
print(" CASE 3: DEMONSTRATION OF UNSEEN NOVEL ANOMALY (High-Frequency Oscillation Pattern)")
print("=" * 80)
# Create an unmodeled novel instance
df_demo_novel = inject_novel_anomalies(df_injected.iloc[[0]].copy(), random_state=7, num_instances=1)
demo_idx_novel = df_demo_novel.index[0]
report_novel = generate_diagnostic_report(demo_idx_novel, df_demo_novel, df_injected_encoded)
print(format_llm_report(report_novel))

## Step 18: Model, Configuration & Metadata Export

In [ ]:
# MODIFIED: Step 18 Model, Configuration & Metadata Export

# 1. Export the Trained Models
joblib.dump(iso_model, 'isolation_forest_model.joblib')
joblib.dump(xgb_clf, 'xgboost_classifier.joblib')

# 2. Export Complete Architecture Metadata and Configuration
pipeline_metadata = {
    "architecture_version": "SkyGuard 2.0 Multi-Tier Complete",
    "feature_cols": feature_cols,
    "num_classes": NUM_CLASSES,
    "classes": ALL_CLASSES,
    "anomaly_classes": ANOMALY_CLASSES,
    "class_to_idx": CLASS_TO_IDX,
    "idx_to_class": {str(k): v for k, v in IDX_TO_CLASS.items()},
    "fusion_config": FUSION_CONFIG,
    "decision_config": DECISION_CONFIG,
    "physics_config": PHYSICS_CONFIG,
    "severity_config": SEVERITY_CONFIG,
    "health_config": HEALTH_CONFIG,
    "export_timestamp": datetime.now().isoformat()
}

with open('pipeline_metadata.json', 'w') as f:
    json.dump(pipeline_metadata, f, indent=4)

print("Successfully Exported Complete SkyGuard Architecture Artifacts:")
print("  - isolation_forest_model.joblib")
print("  - xgboost_classifier.joblib")
print("  - pipeline_metadata.json")

# Google Colab specific download (gracefully skipped in local environments)
try:
    from google.colab import files
    files.download('isolation_forest_model.joblib')
    files.download('xgboost_classifier.joblib')
    files.download('pipeline_metadata.json')
except Exception:
    pass